# Part V: LLM Critique

## Prompt Given to the LLM

> Explain the difference between Monge and Kantorovich optimal transport using a discrete bakery-café problem. Provide the mathematical formulations for both approaches and generate Python code to solve them using the same transportation example.

The LLM gave mathematical formulations for both Monge and Kantorovich transport and then generated code using `scipy.optimize.linprog` for Kantorovich transport and `scipy.optimize.milp` for Monge transport.

I checked both the mathematics and the implementation.

# 1. Does the LLM Enforce the Supply Constraints Correctly?

## Kantorovich Formulation

For Kantorovich transport, the variables are

$$
x_{ij}\ge0,
$$

where $x_{ij}$ is the amount transported from bakery $i$ to café $j$.

The required supply constraint is

$$
\sum_{j=1}^{n}x_{ij}=a_i
$$

for every bakery $i$.

The code constructs these constraints using

```python
for i in range(m):
    row = np.zeros((m, n))
    row[i, :] = 1
```

and sets the corresponding right-hand side equal to

```python
supply[i]
```

Therefore the resulting equality is exactly

$$
x_{i1}+x_{i2}+\cdots+x_{in}=a_i.
$$

Thus the Kantorovich supply constraints are implemented correctly.

## Monge Formulation

For Monge transport, the code introduces binary variables

$$
z_{ij}\in\{0,1\},
$$

where

$$
z_{ij}=1
$$

means that Bakery $i$ sends all of its supply to Café $j$.

The code imposes

$$
\sum_{j=1}^{n}z_{ij}=1.
$$

This ensures that each bakery chooses exactly one café.

The actual shipment is

$$
x_{ij}=a_i z_{ij}.
$$

Therefore, once a bakery chooses one destination, its entire supply is sent there.

Thus the Monge supply rule is also implemented correctly.

---

# 2. Does the LLM Enforce the Demand Constraints Correctly?

## Kantorovich Demand Constraints

The required demand condition is

$$
\sum_{i=1}^{m}x_{ij}=b_j.
$$

The code constructs

```python
for j in range(n):
    row = np.zeros((m, n))
    row[:, j] = 1
```

and sets the right-hand side equal to

```python
demand[j]
```

which produces exactly

$$
x_{1j}+x_{2j}+\cdots+x_{mj}=b_j.
$$

Therefore the Kantorovich demand constraints are correct.

## Monge Demand Constraints

For a Monge assignment, Café $j$ should receive

$$
\sum_{i:T(i)=j}a_i.
$$

Using the binary variables, this becomes

$$
\sum_{i=1}^{m}a_i z_{ij}=b_j.
$$

The code uses

```python
row[:, j] = supply
```

which gives the linear constraint

$$
a_1z_{1j}+a_2z_{2j}+\cdots+a_mz_{mj}=b_j.
$$

Thus the Monge demand constraints are also correct.

There is one minor notation mistake in the written explanation.

It writes

$$
\sum_{i(i)=j}a_i=b_j,
$$

but this should be

$$
\sum_{i:T(i)=j}a_i=b_j.
$$

This appears to be only a typographical error because the code uses the correct mathematical constraint.

---

# 3. Does the LLM Confuse a Transport Map with a Transport Plan?

No.

The LLM correctly distinguishes the two concepts.

For Monge transport, the unknown is a deterministic map

$$
T:\{1,\ldots,m\}\to\{1,\ldots,n\}.
$$

Each bakery has exactly one destination.

In the mixed-integer implementation, the binary matrix

$$
z_{ij}
$$

is simply another representation of this map.

Each row contains exactly one entry equal to $1$.

The corresponding transport quantities are

$$
x_{ij}=a_i z_{ij}.
$$

For Kantorovich transport, the unknown is instead the full transport plan

$$
X=(x_{ij}),
$$

where several entries in one row may be positive.

Thus one bakery may split its supply among several cafés.

The LLM correctly explains that every Monge map generates a Kantorovich plan through

$$
x_{ij}
=
\begin{cases}
a_i, & T(i)=j,\\
0, & T(i)\ne j.
\end{cases}
$$

but not every Kantorovich plan comes from a Monge map.

Therefore, it does not confuse a transport map with a transport plan.

---

# 4. Does the LLM Incorrectly Claim That a Monge Map Always Exists?

No.

The LLM explicitly states that balanced total supply and demand do not guarantee the existence of a Monge map.

This is correct.

Its second example uses

$$
a=(5,5)
$$

and

$$
b=(6,4).
$$

The total masses agree:

$$
5+5=6+4=10.
$$

However, each bakery contains an indivisible mass of $5$.

Under Monge transport, Café 1 could receive

$$
0,\ 5,\text{ or }10
$$

units, but it requires

$$
6.
$$

Similarly, Café 2 requires

$$
4.
$$

Therefore no Monge map exists.

The LLM correctly recognizes this.

The Kantorovich plan

$$
X=
\begin{pmatrix}
5&0\\
1&4
\end{pmatrix}
$$

is feasible because its row sums are

$$
(5,5)
$$

and its column sums are

$$
(6,4).
$$

Thus this example correctly demonstrates that Kantorovich can remain feasible when Monge is infeasible.

---

# 5. Does the LLM Handle the Case $N<M$ Correctly?

This is the main weakness of the response.

The generated solver itself is general and does not assume

$$
N\ge M.
$$

Therefore the code can correctly solve a case with

$$
N<M.
$$

However, neither of the LLM's numerical examples actually has

$$
N<M.
$$

Example 1 has

$$
N=3,
\qquad
M=2,
$$

so

$$
N>M.
$$

Example 2 has

$$
N=2,
\qquad
M=2.
$$

Therefore, although Example 2 correctly demonstrates Monge infeasibility, it does not directly test the specific structural situation

$$
N<M.
$$

To verify that case, I reused the earlier example

$$
N=3,
\qquad
M=4,
$$

with supplies

$$
a=(5,6,4)
$$

and demands

$$
b=(3,4,5,3).
$$

Every café has positive demand.

Since each of the three bakeries can choose only one café, at most three cafés can receive positive mass.

There are four positive-demand cafés.

Therefore no Monge map can exist.

When the LLM-generated solver is applied to this example, the Monge mixed-integer program is correctly reported as infeasible.

The Kantorovich solver returns

$$
\gamma^*
=
\begin{pmatrix}
3&0&2&0\\
0&4&0&2\\
0&0&3&1
\end{pmatrix}
$$

with cost

$$
43.8277238.
$$

Thus the code itself handles $N<M$ correctly, but the original LLM response should have included an explicit $N<M$ example.

---

# Verification of Example 1

Example 1 uses

$$
a=(2,3,5),
$$

$$
b=(5,5),
$$

and

$$
C=
\begin{pmatrix}
2&6\\
3&4\\
6&1
\end{pmatrix}.
$$

A feasible Monge assignment is

$$
T(1)=1,
\qquad
T(2)=1,
\qquad
T(3)=2.
$$

The induced plan is

$$
X=
\begin{pmatrix}
2&0\\
3&0\\
0&5
\end{pmatrix}.
$$

Its row sums are

$$
2,\ 3,\ 5,
$$

and its column sums are

$$
5,\ 5.
$$

Therefore it is feasible.

The cost is

$$
2(2)+3(3)+5(1).
$$

Thus

$$
C=4+9+5=18.
$$

The code returns

$$
C_M^*=18
$$

for Monge and

$$
C_K^*=18
$$

for Kantorovich.

Therefore the LLM's claim is correct.

In this example, allowing mass splitting does not improve the optimal cost.

---

# Verification of Example 2

Example 2 uses

$$
a=(5,5),
$$

$$
b=(6,4),
$$

and

$$
C=
\begin{pmatrix}
2&6\\
3&4
\end{pmatrix}.
$$

The Monge problem is infeasible.

The Kantorovich optimizer returns

$$
X^*
=
\begin{pmatrix}
5&0\\
1&4
\end{pmatrix}.
$$

The row sums are

$$
5,\ 5,
$$

and the column sums are

$$
6,\ 4.
$$

Thus the transport plan is feasible.

Its cost is

$$
5(2)+1(3)+4(4).
$$

Therefore,

$$
C_K^*
=
10+3+16
=
29.
$$

Thus the numerical example is also correct.

---

# Verification of the Optimization Code

The Kantorovich objective used by the code is

$$
\sum_{i=1}^{m}
\sum_{j=1}^{n}
c_{ij}x_{ij},
$$

which is correct.

The Monge mixed-integer objective is

$$
\sum_{i=1}^{m}
\sum_{j=1}^{n}
a_i c_{ij}z_{ij}.
$$

The code implements this using

```python
(cost * supply[:, None]).ravel()
```

which is also correct.

The statement

```python
integrality=np.ones(m * n)
```

together with

```python
Bounds(0, 1)
```

makes each variable integer-valued between $0$ and $1$.

Therefore,

$$
z_{ij}\in\{0,1\}.
$$

Thus the mixed-integer formulation correctly represents the discrete Monge assignment problem.

---

# One Improvement to the Verification Code

Although the optimization code is mathematically correct, it would be safer to verify the returned solutions explicitly.

For a Kantorovich solution $X$, I would check

$$
\sum_j x_{ij}=a_i
$$

and

$$
\sum_i x_{ij}=b_j.
$$

For a Monge assignment $Z$, I would check

$$
\sum_jz_{ij}=1
$$

and

$$
\sum_i a_i z_{ij}=b_j.
$$

This prevents a solver result from being accepted without independently verifying the constraints.

---

# Overall Assessment

The LLM's mathematical explanation is mostly correct.

It correctly:

- distinguishes Monge maps from Kantorovich transport plans;
- imposes the Kantorovich supply constraints correctly;
- imposes the Kantorovich demand constraints correctly;
- imposes the Monge one-destination rule correctly;
- imposes the Monge café-demand constraints correctly;
- recognizes that balanced mass does not guarantee a Monge solution;
- correctly states that every Monge-induced plan is also a Kantorovich plan;
- correctly concludes that

$$
C_K^*\le C_M^*
$$

whenever the Monge problem is feasible.

The main weakness is that the response does not actually demonstrate the requested

$$
N<M
$$

case.

Its second example demonstrates Monge infeasibility, but it has

$$
N=M=2.
$$

After testing the same code on the earlier

$$
N=3<M=4
$$

example, the solver correctly finds that the Monge problem is infeasible while the Kantorovich problem remains feasible through mass splitting.

Therefore, the LLM's reasoning and code are largely reliable in this example, but the response is incomplete with respect to the requested $N<M$ test case.